In [ ]:
# needed to run async in Jupyter

import nest_asyncio
nest_asyncio.apply()

In [ ]:
from google.colab import drive #1
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
folder_to_process = '/content/drive/MyDrive/Indonesia_census' # change to match your folder in Drive

## Extract jpg files from PDF if needed

In [ ]:
%pip install -q pymupdf

In [ ]:
import pymupdf
from pathlib import Path
from tqdm import tqdm

folder_path = Path(folder_to_process)

# Find all PDFs
pdfs = list(folder_path.glob("**/*.pdf"))
print(f"Found {len(pdfs)} PDF files")

# Convert to JPGs
for pdf_file in tqdm(pdfs, desc="Converting PDFs"):
    try:
        doc = pymupdf.open(pdf_file)
        for i, page in enumerate(doc):
            jpg_path = pdf_file.parent / f"{pdf_file.stem}_{i:03d}.jpg"
            if not jpg_path.exists():
                pix = page.get_pixmap(dpi=150)
                img = pix.pil_image()
                img.save(jpg_path)
        doc.close()
    except Exception as e:
        print(f"⚠️  Error processing {pdf_file.name}: {e}")

# Count all images
all_images = list(folder_path.glob("**/*.jpg")) + list(folder_path.glob("**/*.png"))
print(f"📸 Total images to process: {len(all_images)}")

Found 1 PDF files


Converting PDFs: 100%|██████████| 1/1 [00:00<00:00, 18.49it/s]

📸 Total images to process: 21


# Encode all the images together

In [ ]:
import base64

image_messages = []

for image in all_images:
  image_messages.append({"type": "text", "text": image.stem})
  image_messages.append({"type": "image_url", "image_url": { "url": f"data:image/jpeg;base64,{base64.b64encode(image.read_bytes()).decode('utf-8')}"}})




In [ ]:
from google.colab import userdata

# API Configuration
base_url = "https://dashscope-intl.aliyuncs.com/compatible-mode/v1"
api_key = userdata.get('DASHSCOPE_API_KEY')
model = "qwen3-vl-plus"  # or "qwen3-vl-flash" for faster/cheaper

In [ ]:
messages = [
                    {
                        "role": "system",
                        "content": "You are processing a book of printed census data from Indonesia. You goal is to extract the text from all pages and create a single flat JSON representation of the data that can be added to a spreadsheet"
                    },
                {
                  "role": "user",
                  "content": [
                      {
                          "type": "text",
                          "text": "Extract text and return this data as structured JSON."
                      }
                  ]
              }]

In [ ]:
messages[1]['content'].extend(image_messages)

In [ ]:
# @title OCR Processing Functions
import asyncio
import aiohttp
from openai import OpenAI
from tqdm.asyncio import tqdm

client = OpenAI(api_key=api_key, base_url=base_url)



response = client.chat.completions.create(
              model=model,
              messages=messages,
              response_format={"type": "json_object"},
              extra_body={"chat_template_kwargs": {"enable_thinking": True}}
          )

In [ ]:
json_data = response.choices[0].message.content

In [ ]:
import srsly

srsly.write_json('response.json',json_data)

In [ ]:
%pip install -q json-repair

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.1/41.1 kB 2.7 MB/s eta 0:00:00


In [ ]:
# If needed, install once:
# %pip install -q json-repair pandas

import json
import pandas as pd
from pathlib import Path
from json_repair import repair_json

path = Path("response.json")
raw = path.read_text(encoding="utf-8")

def safe_load(text: str):
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        repaired = repair_json(text)
        return json.loads(repaired)

data = safe_load(raw)

if isinstance(data, str):
    data = safe_load(data)

rows = []
for table in data["tables"]:
    for kec in table["data"]:
        for desa in kec["desa"]:
            rows.append({
                "source": data["source"],
                "province": data["province"],
                "table_number": table["table_number"],
                "table_title": table["title"],
                "kecamatan": kec["kecamatan"],
                "desa": desa["name"],
                "laki_laki": desa["laki_laki"],
                "perempuan": desa["perempuan"],
                "jumlah": desa.get("jumlah",''),
            })

df = pd.DataFrame(rows)
df.to_csv("response_flat.csv", index=False)
print("Saved response_flat.csv")

Saved response_flat.csv


In [ ]:
df

,source,province,table_number,table_title,kecamatan,desa,laki_laki,perempuan,jumlah
0,Sensus Penduduk 1961 - NTB-2,NUSATENGARA BARAT,1,JUMLAH PENDUDUK DESA KABUPATEN LOMBOK BARAT ME...,1. Narmada,1. Lembuak,2286,2439,4725
1,Sensus Penduduk 1961 - NTB-2,NUSATENGARA BARAT,1,JUMLAH PENDUDUK DESA KABUPATEN LOMBOK BARAT ME...,1. Narmada,2. Peresah,2289,2492,4781
2,Sensus Penduduk 1961 - NTB-2,NUSATENGARA BARAT,1,JUMLAH PENDUDUK DESA KABUPATEN LOMBOK BARAT ME...,1. Narmada,3. Sedau,1519,1496,3015
3,Sensus Penduduk 1961 - NTB-2,NUSATENGARA BARAT,1,JUMLAH PENDUDUK DESA KABUPATEN LOMBOK BARAT ME...,1. Narmada,4. Batubuta,1404,1554,2958
4,Sensus Penduduk 1961 - NTB-2,NUSATENGARA BARAT,1,JUMLAH PENDUDUK DESA KABUPATEN LOMBOK BARAT ME...,1. Narmada,5. Tanah Beak,1098,1148,2246
...,...,...,...,...,...,...,...,...,...
338,Sensus Penduduk 1961 - NTB-2,NUSATENGARA BARAT,4,JUMLAH PENDUDUK DESA KABUPATEN SUMBAWA MENURUT...,8. Sanggar,1. Talolo,147,141,288
339,Sensus Penduduk 1961 - NTB-2,NUSATENGARA BARAT,4,JUMLAH PENDUDUK DESA KABUPATEN SUMBAWA MENURUT...,8. Sanggar,2. Kore,399,324,723
340,Sensus Penduduk 1961 - NTB-2,NUSATENGARA BARAT,4,JUMLAH PENDUDUK DESA KABUPATEN SUMBAWA MENURUT...,8. Sanggar,3. Boro,361,298,659
341,Sensus Penduduk 1961 - NTB-2,NUSATENGARA BARAT,4,JUMLAH PENDUDUK DESA KABUPATEN SUMBAWA MENURUT...,8. Sanggar,4. Piong,272,249,521


In [ ]:
from google.colab import files

files.download('response_flat.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>